In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"

In [4]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str

agent = create_agent(
    model="ollama:gemma4:e4b",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [5]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config
)

In [6]:
from pprint import pprint

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='4d0e3399-a1f0-475d-bbc6-fef21c6b25de'),
              AIMessage(content='I need more information to fulfill this request. Please provide:\n\n1.  **The address** of the email you would like me to read.\n2.  **The content** or the key points you would like me to include in the reply.', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-04-08T20:01:09.211591Z', 'done': True, 'done_reason': 'stop', 'total_duration': 25340858709, 'load_duration': 5598612125, 'prompt_eval_count': 114, 'prompt_eval_duration': 6413339167, 'eval_count': 385, 'eval_duration': 13088696628, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019d6eae-d156-7bd2-

In [7]:
print(response['__interrupt__'])

KeyError: '__interrupt__'

In [8]:
# Access just the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

KeyError: '__interrupt__'

## Approve

In [9]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='4d0e3399-a1f0-475d-bbc6-fef21c6b25de'),
              AIMessage(content='I need more information to fulfill this request. Please provide:\n\n1.  **The address** of the email you would like me to read.\n2.  **The content** or the key points you would like me to include in the reply.', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-04-08T20:01:09.211591Z', 'done': True, 'done_reason': 'stop', 'total_duration': 25340858709, 'load_duration': 5598612125, 'prompt_eval_count': 114, 'prompt_eval_duration': 6413339167, 'eval_count': 385, 'eval_duration': 13088696628, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019d6eae-d156-7bd2-

## Reject

In [11]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No please sign off - Your merciful leader, Seán."
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='4d0e3399-a1f0-475d-bbc6-fef21c6b25de'),
              AIMessage(content='I need more information to fulfill this request. Please provide:\n\n1.  **The address** of the email you would like me to read.\n2.  **The content** or the key points you would like me to include in the reply.', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-04-08T20:01:09.211591Z', 'done': True, 'done_reason': 'stop', 'total_duration': 25340858709, 'load_duration': 5598612125, 'prompt_eval_count': 114, 'prompt_eval_duration': 6413339167, 'eval_count': 385, 'eval_duration': 13088696628, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019d6eae-d156-7bd2-

In [12]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

KeyError: '__interrupt__'

## Edit

In [10]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='4d0e3399-a1f0-475d-bbc6-fef21c6b25de'),
              AIMessage(content='I need more information to fulfill this request. Please provide:\n\n1.  **The address** of the email you would like me to read.\n2.  **The content** or the key points you would like me to include in the reply.', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-04-08T20:01:09.211591Z', 'done': True, 'done_reason': 'stop', 'total_duration': 25340858709, 'load_duration': 5598612125, 'prompt_eval_count': 114, 'prompt_eval_duration': 6413339167, 'eval_count': 385, 'eval_duration': 13088696628, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019d6eae-d156-7bd2-